# Exploring GNSS-R data from Rongowai

*Authors:* Prof. Matthew Wilson and Xander Cai

*Geospatial Research Institute Toi Hangarau, University of Canterbury, Christchurch, New Zealand*

*Last Updated: 10 April 2026. Version: 1.0.0*

This Jupyter notebook provides an introduction to Rongowai GNSS-R data, collected from the sensor onboard an Air New Zealand Q300 aircraft (tail number: [ZK-NHA](https://www.flightradar24.com/data/aircraft/zk-nfa)), which operates several times a day across the New Zealand regional network. Since its launch in 2022 there have been more than 7,000 flights, making for an extremely rich dataset. For further information about the development of the mission, check out the [Rongowai website](https://spoc.auckland.ac.nz/). For a deep dive on GNSS-R (Geographic Navigation Satellite System - Reflectometry), see the open access book *Fundamentals and Applications of GNSS Reflectometry* by [Yang & Wang (2025)](https://doi.org/10.1007/978-981-96-4554-1).

The aim of the notebook is to provide an interactive walk through the data to gain familiarity with it. Code is provided to import the data (from L1 netcdf files), save it in geospatial formats, and visualise it as maps and plots. After completing the notebook, you should feel confident enough to be able to start developing analyses using the data.


## Getting started

The cell below will check your environment for required packages. Please run this to install any missing packages. Alternativaly you can create an environment using the ``environment.yml`` file provided, then check the status here. Setting up an environment from scratch will take several minutes, but it is worth doing this to avoid dependency clashes with your other Python work.

```bash
conda env create --file environment.yml
conda activate rongowai
python -m ipykernel install --user --name rongowai --display-name "Python (Rongowai)"
```

In [ ]:
import setup

# --- CONFIGURATION ---
TARGET_PYTHON = "3.14"  
REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "geopandas": "geopandas",
    "xarray": "xarray",
    "netCDF4": "netcdf4",
    "leafmap": "leafmap",
    "pydeck": "pydeck",
    "tqdm": "tqdm",
    "numpy": "numpy",
    "shapely": "shapely",
    "geopy": "geopy",
    "scipy": "scipy",
    "itables": "itables",
    "lonboard": "lonboard",
    "matplotlib": "matplotlib",
    "json": "json",
    "ipywidgets": "ipywidgets",
    "openpyxl": "openpyxl",
}

setup.check_python_environment(REQUIRED_PACKAGES, TARGET_PYTHON)


Import packages and setup data paths:

In [ ]:
import rongowai_helpers as rongowai # required for importing data and the plotting functions used

import pandas as pd
import geopandas as gpd
import xarray as xr
import netCDF4 as nc

import re
from IPython.display import display, clear_output, HTML
from itables import show
import itables.options as opt
import ipywidgets as widgets

from pathlib import Path
import pydeck as pdk
from tqdm import tqdm
import random

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import os

# uncomment this to make the random numbers deterministic (useful for exact replication, e.g. for debugging and testing)
#seed = 0
#random.seed(seed)

verbose = True

data_root = Path("./data")
L1_data = data_root / "L1"
gpkg_data = data_root / "GPKG"
aoi = data_root / "AoI.gpkg"

## Rongowai L1 data structure

Rongowai L1 data are available from PODAAC here: https://podaac.jpl.nasa.gov/dataset/RONGOWAI_L1_SDR_V1.0. Individual files can be downloaded directly from the [directory listing](https://cmr.earthdata.nasa.gov/virtual-directory/collections/C2784494745-POCLOUD/temporal), or programmatically using the [Data Subscriber Python package on Github](https://github.com/podaac/data-subscriber) (this requires an [Earthdata login](https://urs.earthdata.nasa.gov/)).

Here, we will use some previously downloaded data for exploratory analysis. The code below checks files in the data directory, views a table, then chooses a random file (edit this code if you want to select a specific file).

**Get a data frame of available files and view:**

In [ ]:
file_list = [f.as_posix() for f in L1_data.glob("*.nc")]
df_files = rongowai.parse_rongowai_files(file_list)

show(
    df_files.drop(columns=["datetime_utc"]),  # datetime_utc is a helper column; drop for display
    caption="Rongowai flight files",
    classes="display compact",
    columnDefs=[{"width": "350px", "targets": 0}],  # widen filename column
    paging=True,
    pageLength=15,
    layout={"top": "searchBuilder"},   # adds per-column filter builder
)

**Exercise**: 

- You can use the custom filters to find flights on to/ from a specific airport, or on a particular day.

**Select a random file and open the netcdf:**

(edit this code if you want to select a specific file)

In [ ]:
print(f"{len(file_list)} available L1 files.")
selected_file = random.choice(file_list)
print(f"-> file {Path(selected_file).name} selected.")
ds = xr.open_dataset(selected_file, engine="netcdf4")
ds


We will extract data from this netcdf file, available through ``ds`` for plotting, but to understand the data structure, it is useful to refer to the data dictionary.

**View the data dictionary:**

The L1 [data dictionary](https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-docs/cygnss/open/L1/docs/L1_Dict_v2_4.xlsx) provides detailed information about the data within the netcdf files, and is an essential companion to interacting with the data (this information is also in the netcdf). Let's load and view this now.

In [ ]:
# --- Load and display data dictionary ---------------------------------------------------
df_dict = pd.read_excel("L1_Dict_v2_4.xlsx", dtype=str).fillna("<none>")
rongowai.showDataDict(df_dict)

**Exercise**: It is worth spending a bit of time exploring these data. In particular: 
- use the Dimensions filter to see which variables are in which dimension. Note the "Sample" dimension is key and ties all observations together. 
- check these variables: ``ddm_ant``, ``ddm_snr``, ``surface_reflectivity``, ``surface_reflectivity_peak`` (these will be used for plotting below).
- variables ``sp_*`` are related to the specular point data (dimensions: ``sample``, ``ddm``), ``ac_*`` refer to the aircraft data (dimensions: ``sample``), and ``tx_*`` refer to the transmitter (i.e. the GPS satellites).

For more detailed information about the data, please see the [documentation on PODAAC](https://podaac.jpl.nasa.gov/dataset/RONGOWAI_L1_SDR_V1.0).

## Extract and explore data

To map the L1 data, we need to extract the spatial location vectors from the nc file. We can then export the data for viewing or analysing in GIS. The functions referred to below will extract the two dimensional data (dimensions: ``sample``, ``ddm``, as specified in the data dictionary, and save them to a long-format database table (geopackage). It will also obtain data related to the flight, including the sensor location of each observation. Note that this does not include the 4D data (DDMs), which need to be extracted from the nc file individually.


**Process the netcdf data into a geopackage:**

The cell below will pull in the 2D data from the netcdf file (dimensions: ``sample``, ``ddm``) and save to a geopackage file that can be easily loaded in QGIS (this will take several seconds or longer for large files).

In [ ]:
gpkg_file = Path(selected_file).parent.parent / f"{os.path.basename(selected_file)}.gpkg"

_ = rongowai.l1_to_gpkg_single(selected_file, gpkg_file)

**Load the spatial data into Python and view a table:**

In [ ]:
gdf = gpd.read_file(gpkg_file, layer = 'spatial') # 2D point data for each GNSS-R return
gdf_ac = gpd.read_file(gpkg_file, layer = 'ac') # point data for the aircraft's position at each timestamp
gdf_flightvector = gpd.read_file(gpkg_file, layer = 'flightvector') # a polyline of the flight path

rongowai.showGeoDataFrame(gdf, caption=gpkg_file.name, df_dict=df_dict, maxBytes=10240) # set maxBytes to 0 to disable size limit (but it will take a couple of minutes to load if the file is large)

**Exercise**:

- Spend a bit of time exploring this table to build your familiarity. Cross reference it with the data dictionary you openned above.

Note that:
- There are up to **20** values for each ``sample``, corresponding to ``ddm`` index 0 to 19. Each ``sample`` corresponds to a set of simultaneous observations recorded at ``ddm_timestamp_utc``.
- The ``ddm_ant`` variable specifies whether the data are LHCP (2) or RHCP (3). LHCP is always in ``ddm`` index 0 to 9, and RCHP is always in ``ddm`` index 10 to 19. The data are **paired**: i.e. ``ddm`` index 0 is paired with ``ddm`` 10, for the same location, GPS satellite (``sv_num``) and time (``ddm_timestamp_utc``), index 1 is paired with 11, etc. You can see this in the data table above by selecting the variables  ``sample``, ``sp_lat``, ``sp_lon``, ``ddm`` and ``ddm_ant``, then comparing the locations for paired data. 
  
You can see the ``sample`` and ``ddm_timestamp_utc`` in the aircraft track data, already loaded into ``gdf_ac``:

In [ ]:
rongowai.showGeoDataFrame(gdf_ac, caption=gpkg_file.name, df_dict=df_dict, maxBytes=10240) # set maxBytes to 0 to disable size limit (but it will take a couple of minutes to load if the file is large)

Note that each ``sample`` in the ``gdf_ac`` geodataframe is unique: it denotes the set of observations recorded at ``ddm_timestamp_utc``, which is in seconds since the start.

## Map the extracted data

It is useful to map the location of the data for further analysis: we can do this based on the coordinates recorded for the aircraft (``ac_lon``, ``ac_lat`` and ``ac_alt``) and each specular point observation (``sp_lon``, ``sp_lat`` and ``sp_alt``). These have already been loaded into a GeoDataFrame above.

The map below will default to the ``surface_reflectivity_peak`` variable, with the radius of each point set to the value of ``fresnel_minor``. 

In [ ]:
# Select only LHCP returns for mapping (note that LHCP and RHCP returns are for the identical locations and timestamps, 
# so the points would be stacked on top of each other if both were plotted). Feel free to re-run this cell with 'ddm_ant == 3' to select only RHCP returns for comparison).
filtered_gdf = gdf.query('ddm_ant == 2')

In [ ]:
# Map the filtered data
rongowai.map_rongowai_interactive(
    filtered_gdf,
    gdf_points=gdf_ac,
    gdf_track=gdf_flightvector,
)

**Exercise**: 

Here are some things to explore:
- Try adjusting the colour range to see details more clearly (use statistics from the tables above to help). 
- Map ``ddm_snr``, and notice how this changes depending on the surface type observable in the satellite image.
- The surface type is encoded in the ``sp_surface_type`` variable: visualise this using a categorial colourmap like ``tab10``.
- Map the incidence angle of the signal, via ``sp_inc_angle``: notice how signals further away from the aircraft have a higher angles.
- Map the ``fresnel_minor`` and ``fresnel_major`` columns: how are they related to ``sp_inc_angle``? 
- Plot the GPS satellite ID (``sv_num``) using a categorical colourmap like ``tab20``. Notice how each satellite forms a "track" of observations in the data.
- Click on points to view the attribute data.

**Estimate the Fresnel zones**

The map above is of the specular point locations, with the points scaled according to ``fresnel_minor``. To estimate where the signal has come from, we can do better than this with a bit of geometric wrangling. The code below will produce polygons representing the Fresnel zone, based on ``fresnel_minor``, ``fresnel_major`` and ``fresnel_orientation``, while accounting for the movement of the aircraft and GPS satellites during the 1 s observation (via ``sp_vel_x`` and ``sp_vel_y``). The code below will add the layers "fresnel" and "fresnel_integration" to your geopackage file. These represent the "instantaneous" Fresnel zones at the specular point locations, and the stretched the Fresnel zones caused by aircraft/ satellite movement, respectively.

*Warning: this is slow and will take a several minutes to complete*.

In [ ]:
gdf_fresnel = rongowai.fresnel(gdf)
gdf_fresnel.to_file(gpkg_file, layer='fresnel', driver="GPKG")
gdf_fresnel_integration = rongowai.fresnel_integration(gdf)
gdf_fresnel_integration.to_file(gpkg_file, layer='fresnel_integration', driver="GPKG")

In [ ]:
# To reload the Fresnel layers from the GPKG (instead of re-computing them) for display in the next cell, uncomment the following two lines:
#gdf_fresnel = gpd.read_file(gpkg_file, layer = 'fresnel') 
#gdf_fresnel_integration = gpd.read_file(gpkg_file, layer = 'fresnel_integration') 

In [ ]:
filtered_gdf_fresnel = gdf_fresnel.query('ddm_ant == 2')

rongowai.map_rongowai_interactive(
    filtered_gdf_fresnel,   # polygon GeoDataFrame — detected automatically
    gdf_points=gdf_ac,
    gdf_track=gdf_flightvector,
)

Notice how the Fresnel zone size and orientation changes depending on the satellite/ aircraft viewing geometry, with the smallest Fresnel zones located closest to the flight track (with low values of `sp_inc_angle`). This information is important for inferring surface properties from the data.

**Exercise**:

- Change the call in the cell above to filter from the ``gdf_fresnel_integration`` geodataframe rather than ``gdf_fresnel``. How do they compare? In this data, the polygons representing the Fresnel zones are "stretched" as the sensor records over a period of 1 s, during which time both the aircraft and satellites are moving. 

## Exploring data

Note all data recorded by the sensor is useable for analysis (e.g. those that have a high incidence angle will not be useful). Run the cell below to filter the Rongowai data to select high quality observations, then use the interactive plot to explore the data values. 

In [ ]:
# define some reasonable filters to select quality Rongowai observations
filter_dict = {
        "coherence_metric":    (1, 9999), # To select coherent returns, set the lower bound to 1 (since coherence_metric is 0 for incoherent returns and >1 for coherent returns). 
        "sp_inc_angle":        (0, 60),  # Specular incidence angle under 60 degrees should be good quality, but you can adjust this threshold as needed
        "ddm_snr":             (-10.0, 9999), # DDM SNR over -10 dB should be good quality, but you can adjust this threshold as needed
        "ddm_ant":             (2, 2),  # LHCP only
        "sp_surface_type":     (0, 7),  # -1 = ocean, 1 = artifical, 2 = barely vegetated, 3 = inland water, 4 = crop, 5 = grass, 6 = shrub, 7 = forest
    }

# apply the filter to the data (not that plot_rongowai_interactive can do this filtering internally as well, but we'll do it here as we'll us the data again below)
filter_mask = pd.Series(True, index=gdf.index)
for col, (lo, hi) in filter_dict.items():
    if col in gdf.columns:
        filter_mask &= gdf[col].between(lo, hi) | gdf[col].isna()
filtered_gdf = gdf[filter_mask].copy()

rongowai.plot_rongowai_interactive(
    filtered_gdf,
    #filters=filter_dict, # you can also apply the filters directly in the plotting function, but we already applied them to create filtered_gdf
    default_x="ddm_snr",
    default_y="surface_reflectivity_peak",
    default_color="sp_surface_type",
    df_dict=df_dict,
)

**Exercise**: Here are some questions to explore and help your understanding:
- How does surface reflectivity change depending on SNR? How does it depend on the surface type? What types of surface have the highest reflectivity?
- How does the incidence angle affect the SNR and surface reflectivity? How does it affect the Fresnel zone size?

## Plotting the DDMs for different surfaces

The 2D data above have been derived from the observations encoded as delay-Doppler maps (DDMs), which for Rongowai data are binned into 5x40 arrays. The code below will pick random observation points from the filtered data (one each for each surface type, ``sp_surface_type``), then extract the DDM for that observation from the netcdf dataset, ``ds``. Play around with the interactive plot to get familiar with the DDM data. Make sure you click "Redraw sample" a few times to see a different set of DDMs. 

In [ ]:
rongowai.plot_ddm_interactive(
    filtered_gdf,
    ds,
    ddm_var="surface_reflectivity",
    surface_type_labels={
        -1: "-1: Ocean", # note: filtered above but included here for completeness
        1: "1: Artificial",
        2: "2: Barely vegetated",
        3: "3: Inland water",
        4: "4: Crop",
        5: "5: Grass",
        6: "6: Shrub",
        7: "7: Forest",
    },
    metadata_cols=["sp_lat", "sp_lon", "sp_inc_angle", "ddm_snr", "coherence_state",
                   "surface_reflectivity_peak"],
)

**Exercise**: 

- Just for fun, try experimenting with the filter used on the data above. What do the DDMs look like for incoherent data (``coherence_metric`` < 1), or for high incidence angles (``sp_inc_angle`` > 80)? 


## Summary and next steps

You should now have a good idea about the format of the Rongowai L1 data and be able to get started with analysis using the data. In this notebook we:
- viewed the data dictionary for the data format
- imported data from a L1 netcdf and saved the 2D data as a geopackage for loading in GIS software
- visualised these data using an interactive map
- estimated and visualised the Fresnel zone areas for each specular point
- viewed DDMs for each land surface type encoded within the data

From here: one of the interesting ways to analyse the data is to use other data of the land surface (e.g. surface slope from an elevation model, soil type etc.). Because the data are geospatially encoded, it is relatively straightforward to combine geospatial datasets, for example by extracting values from supplementary rasters using a point or polygon query. Further, many locations have numerous observations available, enabling comparison of values over time. The challenge here observations are never exactly aligned; a good approach to handling this is to filter the Rongowai data (as above) and assign selected values to a grid, for example to obtain a monthly mean surface reflectance.

## Further reading

1. Bai, D., Ruf, C.S. and Moller, D., 2025. Calibration of the polarimetric GNSS-R sensor in the rongowai mission. IEEE Transactions on Geoscience and Remote Sensing. https://doi.org/10.1109/TGRS.2025.3558131
1. Carreno-Luengo H, Ruf CS, Gleason S, Russel A. Latest progress on rongowai polarimetric GNSS-R airborne mission. In *IGARSS 2024-2024 IEEE International Geoscience and Remote Sensing Symposium* 2024 Jul 7 (pp. 6828-6830). IEEE. https://doi.org/10.1109/IGARSS53475.2024.10642216
1. Peng, J., Cardellach, E., Li, W., Ribó, S. and Rius, A., 2025. Impact of right-hand polarized signals in GNSS-R water detection algorithms. *IEEE Journal of Selected Topics in Applied Earth Observations and Remote Sensing*, 18, pp.5646-5655.
1. Moller, D., Orzel, K., Andreadis, K. and Wilson, M., 2024, July. An Airborne GNSS-R Driven Low-Latency Flood Assessment Development. In *IGARSS 2024-2024 IEEE International Geoscience and Remote Sensing Symposium* (pp. 1370-1373). IEEE.
1. Moller, D., Wilson, M., Datta, R., O'Brien, A., Linnabary, R. and Ruf, C., 2022, July. Rongowai: A pathfinder NASA/NZ GNSS-r initiative supporting SDG-15-life on land. In *IGARSS 2022-2022 IEEE International Geoscience and Remote Sensing Symposium* (pp. 4212-4215). IEEE. https://doi.org/10.1109/IGARSS46834.2022.9884397
1. Wilson, M., Han, S.C., Khaki, M., Yeo, I.Y., Townsend, G., Yuan, F., Nguyen, M., Cai, X., Ash, C. and Moller, D., 2025, August. Satellite Sensing Into Agricultural Practices: Cal/Val Campaign of Satellite, Airborne and Ground GNSS-Reflectometry of Soil Moisture. In *IGARSS 2025-2025 IEEE International Geoscience and Remote Sensing Symposium* (pp. 1117-1120). IEEE.
1. Wilson, M., Savarimuthu, S., Moller, D., Cai, X. and Ruf, C., 2024, April. Estimation of soil moisture from Rongowai GNSS-R using machine learning. In *2024 International Conference on Machine Intelligence for GeoAnalytics and Remote Sensing (MIGARS)* (pp. 1-4). IEEE.
1. Xu, Z., Wang, X., Xia, J., Sun, Y., Liu, C., Wang, Z., Tian, Y., Qiu, T. and Wang, D., 2026. GATENet: Spectral-Auxiliary Attention Network for Airborne GNSS-R Based Topographic Estimation. *IEEE Geoscience and Remote Sensing Letters*. https://doi.org/10.1109/LGRS.2026.3668630


